# NexaTel CDR Pipeline — Task 2: Cleanse CDR (Silver)

> **Industry context — NexaTel Communications**  
> Raw CDR records from the network switches contain quality issues:  
> missing subscriber IDs, negative durations, and malformed identifiers.  
> This notebook is **Task 2** in the nightly Lakeflow Job. It reads the raw CDR,  
> applies cleansing rules, and writes a validated **silver_cdr** table.

**Lakeflow Job DAG position:** `ingest_cdr` → **`cleanse_cdr`** → `aggregate_usage`  
**Depends on:** `ingest_cdr` (run-if condition: **All succeeded**)

In [ ]:
# ── CELL 1: Read job parameters and upstream task values ──────────────────────
# Trainer: show how Task 2 picks up the billing_month that Task 1 stored.
# This avoids hardcoding dates and makes the pipeline reusable.

dbutils.widgets.text("billing_month",  "2024-12", "Billing Month (YYYY-MM)")
dbutils.widgets.text("source_catalog", "trainer_demo", "Source Catalog")
dbutils.widgets.text("source_schema",  "demo_11",       "Source Schema")

billing_month  = dbutils.widgets.get("billing_month")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")

# Try to read the total_raw_records task value set by Task 1
try:
    total_raw = int(dbutils.jobs.taskValues.get(
        taskKey="ingest_cdr", key="total_raw_records"
    ))
    print(f"Task value from ingest_cdr: total_raw_records = {total_raw:,}")
except Exception:
    total_raw = None
    print("[Standalone mode] task values not available — running without upstream context")

print(f"Processing billing month: {billing_month}")

In [ ]:
# ── CELL 2: Read raw CDR and filter to billing month ─────────────────────────

from pyspark.sql import functions as F

year, month = billing_month.split("-")

raw_cdr = (
    spark.read.table(f"{source_catalog}.{source_schema}.raw_cdr")
    .filter(
        (F.year("call_start_ts")  == int(year)) &
        (F.month("call_start_ts") == int(month))
    )
)

raw_count = raw_cdr.count()
print(f"Raw records for {billing_month}: {raw_count:,}")

In [ ]:
# ── CELL 3: Apply cleansing rules ─────────────────────────────────────────────
# Trainer: walk through each rule and explain WHY it matters for NexaTel billing.
#
# Rule 1 — Drop records with NULL subscriber_id  : cannot bill an unknown customer
# Rule 2 — Drop records with malformed subscriber_id: regulatory requirement
# Rule 3 — Drop records where duration_seconds <= 0: network glitch, not a real call
# Rule 4 — Derive call_end_ts                      : required by the billing system
# Rule 5 — Standardise termination_code to UPPER   : inconsistent casing from switches

VALID_SUBSCRIBER_PATTERN = r"^SUB-\d{5}$"

silver_cdr = (
    raw_cdr
    # Rule 1: drop missing subscriber
    .filter(F.col("subscriber_id").isNotNull())
    # Rule 2: drop malformed subscriber IDs
    .filter(F.col("subscriber_id").rlike(VALID_SUBSCRIBER_PATTERN))
    # Rule 3: drop zero / negative durations
    .filter(F.col("duration_seconds") > 0)
    # Rule 4: derive call_end_ts from start + duration
    .withColumn(
        "call_end_ts",
        F.col("call_start_ts") + F.expr("INTERVAL duration_seconds SECONDS")
    )
    # Rule 5: standardise termination code casing
    .withColumn("termination_code", F.upper(F.col("termination_code")))
    # Add audit columns
    .withColumn("_processed_at", F.current_timestamp())
    .withColumn("_billing_month", F.lit(billing_month))
)

silver_count = silver_cdr.count()
rejected     = raw_count - silver_count
reject_pct   = rejected / raw_count * 100 if raw_count > 0 else 0

print(f"\n=== Cleansing summary for {billing_month} ===")
print(f"  Raw records       : {raw_count:>8,}")
print(f"  Silver records    : {silver_count:>8,}")
print(f"  Rejected          : {rejected:>8,}  ({reject_pct:.1f}%)")
print()
silver_cdr.select(
    "cdr_id", "subscriber_id", "call_type",
    "duration_seconds", "call_end_ts", "termination_code"
).show(5)

In [ ]:
# ── CELL 4: Write Silver CDR table ────────────────────────────────────────────
# Trainer: note MERGE INTO here — we upsert so that re-running the job is safe
# (idempotent). This is critical for the retry demos in the automatic restarts unit.

target_table = f"{source_catalog}.{source_schema}.silver_cdr"

# Create or replace for demo simplicity;
# in production use MERGE INTO for idempotency
(
    silver_cdr
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"✓ Written {silver_count:,} records to {target_table}")

# Share results downstream
try:
    dbutils.jobs.taskValues.set(key="silver_record_count", value=silver_count)
    dbutils.jobs.taskValues.set(key="rejected_count",      value=rejected)
    print(f"  task value: silver_record_count = {silver_count:,}")
    print(f"  task value: rejected_count       = {rejected:,}")
except Exception:
    print(f"[Standalone mode] Would set silver_record_count={silver_count:,}, rejected_count={rejected:,}")

print("\n✓ Task 2 (cleanse_cdr) complete.")
dbutils.notebook.exit("SUCCESS")